<img src="https://courseware.decoded.com/images/decoded/logo-decoded.png" align="left" width="100px"/>

# Day 4 — Bitemporal loading and Airflow orchestration

<small>Data Creators › Day 4 — anchor notebook</small>

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>

## 🗺️ Table of contents

| Section | What happens |
|:---|:---|
| [How to use this notebook](#how-to-use) | Start here · start the Airflow stack first |
| [Welcome](#welcome) | Agenda |
| [Bitemporal loading](#bitemporal) | Two clocks · the demo · the macro · which clock answers which question |
| [Orchestration and Friday](#orchestration) | The DAG · the capstone preview |
| [Airflow](#airflow) | Components · the demo stack · operators, >>, callbacks, blueprints · a controlled failure |
| [Challenge 10](#challenge) | Launch box · support table · extension questions |
| [CI/CD and deployment](#cicd) | GitOps · Harness · Datadog · secrets |
| [Release decision and the failure note](#exercises) | Activities 10 and 11 |
| [Friday preparation](#friday) | The checklist |
| [Questions from the room](#questions) | Live Q&A, answers, sketches |

<a id="how-to-use"></a>

## How to use this notebook

- This is the anchor for Day 4: the agenda, every concept with its diagram, the whiteboard scripts, the activity briefs and the pointers into the learners' challenges. Keep it on the projector; switch to a terminal or the browser when a 🖥️ DEMO box says so.
- Run the **Setup** cell first. Cells that start with `!` run in a shell in this notebook's folder with the demo environment on `PATH`.
- 🖊️ Whiteboard blocks are folded: open one when you reach it, ask the questions in order, write what the room says, add what is missing.
- The last section, **Questions from the room**, is yours to type in during the day: a table for questions, an empty cell to answer with SQL or Python, and `quick_diagram(...)` to sketch a diagram inline before reaching for Excalidraw.
- Learners can be given this notebook as it is; nothing in it is instructor-only. The instructor-only timing notes are in `run-sheet.md` beside it.
- Before the day: `bash demos/airflow_local/run.sh start` (Docker Desktop running; the image is already built) and `bash demos/bitemporal/demo.sh` once. The Airflow UI is http://localhost:8080.

In [ ]:
# Setup: the demo environment on PATH, DuckDB and pandas for showing results, a quick diagram helper
import os, sys, subprocess, textwrap
from pathlib import Path
from IPython.display import Image, Markdown, display

VENV = Path(os.environ.get("CG_DXP_VENV", Path.home() / ".venvs" / "cg-dxp"))
os.environ["PATH"] = ":".join([str(VENV / "bin"), str(Path.home() / ".local" / "bin"),
                               "/Applications/Docker.app/Contents/Resources/bin", "/opt/homebrew/bin", os.environ["PATH"]])
os.environ.setdefault("DBT_SEND_ANONYMOUS_USAGE_STATS", "false")
HERE = Path.cwd()                      # courseware/day4-*/
COMMON = HERE.parent / "common"
assert COMMON.exists(), "Open this notebook from its own folder (courseware/day4-*)"

def quick_diagram(mermaid: str, name: str = "scratch"):
    """Render a Mermaid diagram typed live and show it inline (needs node; first use downloads mermaid-cli)."""
    src = HERE / f"{name}.mmd"; out = HERE / f"{name}.png"
    src.write_text(textwrap.dedent(mermaid).strip() + "\n")
    cfg = COMMON / "diagrams" / "mermaid-config.json"
    r = subprocess.run(["npx", "-y", "-p", "@mermaid-js/mermaid-cli@11.17.0", "mmdc", "-i", str(src), "-o", str(out), "-c", str(cfg), "-b", "white", "-s", "2"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-800:]); return
    display(Image(filename=str(out)))

def sql(query: str, db: Path, read_only: bool = True):
    """Run a query against a DuckDB file and show it as a table."""
    import duckdb, pandas as pd
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
    con = duckdb.connect(str(db), read_only=read_only)
    try:
        return con.execute(query).df()
    finally:
        con.close()

print("✅ Day 4 setup:", subprocess.run(["dbt", "--version"], capture_output=True, text=True).stdout.split("\n")[1].strip() if subprocess.run(["which", "dbt"], capture_output=True).returncode == 0 else "dbt not on PATH: run common/env/setup-env.sh")

<a id="welcome"></a>

## Welcome

### Today

| Time | Block | Type | Where |
|---|---|---|---|
| 09:00–09:45 | Bitemporal loading | Demo + shareback | Anchor › Bitemporal (diagram, demo cells, the macro); a guest or volunteer's screen |
| 09:45–10:00 | Orchestration and capstone orientation | Demo | Anchor › The DAG diagram; Friday preview |
| 10:00–10:15 | Break | Break |  |
| 10:15–10:45 | Airflow demonstration | Demo | Anchor › Airflow cells; the Airflow UI at localhost:8080 |
| 10:45–12:00 | Challenge 10 — Airflow practical | Challenge | Anchor › Challenge box; learners in challenges/v1/03-airflow-orchestration |
| 12:00–13:00 | Lunch | Break |  |
| 13:00–13:30 | Continue Airflow practical, regroup at 13:30 | Challenge + shareback | Anchor › Challenge box |
| 13:30–14:00 | CI/CD and deployment context | Demo | Anchor › CI/CD (diagram), Harness walkthrough, Datadog monitors |
| 14:00–14:15 | Break | Break |  |
| 14:15–14:35 | Release decision in pairs | Challenge + shareback | Anchor › Activity 10 card |
| 14:35–14:50 | Explain a failure and repair | Challenge + shareback | Anchor › Activity 11 card and template |
| 14:50–15:00 | Friday preparation and closing questions | Shareback | Anchor › Friday checklist |

Morning: history (bitemporal loading) and orchestration (Airflow) with a long practical. Afternoon: the deployment picture, then two short exercises that turn a failed run into a decision and a note. Close by checking Friday's starting line.

<a id="bitemporal"></a>

## Bitemporal loading

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">The source corrects Monday's value on Wednesday. What was true on Monday, and what did we believe on Tuesday?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Two clocks on every row: business time (when it was true) and system time (when the warehouse knew it). Nothing is overwritten; corrections add versions.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Audit, reproducible reports and 'as was' questions all need the second clock; a status flag or an overwrite loses it.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Three loads on a tiny associate table: a day-1 file, a day-2 file with a change and a defect, a corrected re-delivery. Then four queries.</div></div></div>

<details open><summary><strong>🖊️ Whiteboard — loading strategies and two clocks</strong> (ask, write what the room says, add what is missing)</summary>
<table style="width:100%;border-collapse:collapse;margin:8px 0;color:#1a1a1a"><tr><th style="text-align:left;padding:6px 10px;background:#F3F3F3">Ask</th><th style="text-align:left;padding:6px 10px;background:#F3F3F3">Expect, then add</th></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>How can new data land in a table?</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Truncate and reload; insert-overwrite; SCD type 2 (effective start and end dates keep history). Bitemporal is SCD2 with a second clock.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>What are the two clocks?</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Business time: when the fact was true in the world (the business date). System time: when the warehouse loaded or believed it. Separate columns for start and end of each, plus a current indicator and a row hash.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Full, incremental, balance?</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Full: the source sends the whole book of record each run (expensive: every row re-versioned). Incremental: only new or changed records with a business key; a known key with new content ends the previous version. Balance: for monetary balances. Some models need a custom strategy.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Why does a correction matter?</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Because 'as was' must still answer what the report said on Tuesday, and 'as of' must give the corrected Monday value today. Overwriting destroys the first; a status flag cannot express either.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>UTC or local?</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>The platform stamps system time in UTC; a run at 03:00 Pacific for the previous business day shows yesterday's business date. Always say which clock a date column is.</td></tr></table></details>

![One associate across three loads: four rows, nothing overwritten](../common/diagrams/png/d4-bitemporal-rows.png)

- Day 2's changed record closes the day-1 row's system clock and carries it forward with its business clock ended, then inserts the new version.
- The corrected re-delivery for the same business date closes only the system clock of the defective row and inserts the corrected one.
- As of Monday with today's knowledge: the carried-forward row. As was on Tuesday: the defective row. As is: the corrected row.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; three loads on DuckDB, then the four questions</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">The cell runs <code>demos/bitemporal/demo.sh</code>: seed, load day 1, load day 2, load the correction, then prints the history, as-is, as-of and as-was queries. The strategy is a custom dbt incremental strategy in <code>demos/bitemporal/macros/bitemporal_load.sql</code>, written to be read: the paved path's <code>bitemporal_load_v2</code> macro does the same job with more strategies.</div></div>

In [ ]:
!bash demos/bitemporal/demo.sh 2>&1 | sed -n '/-- Full history/,$p'

In [ ]:
sql("""select associate_id, job_title, tenure_months, dl_business_start_dt, dl_business_end_dt, dl_system_start_ts, dl_system_end_ts, dl_current_ind
          from training_demo.dim_associate where associate_id = 1001 order by dl_system_start_ts, dl_business_start_dt""", HERE / "demos/bitemporal/warehouse/bitemporal.duckdb")
# The audit trail for one associate. Point at each row: which load wrote it, and which clock closed it.

In [ ]:
sql("select associate_id, job_title, tenure_months from training_demo.dim_associate where dl_business_start_dt <= date '2026-11-02' and coalesce(dl_business_end_dt, date '9999-12-31') >= date '2026-11-02' and dl_system_end_ts is null order by 1", HERE / "demos/bitemporal/warehouse/bitemporal.duckdb")
# AS OF 2 November with today's knowledge: business clock covers the date, system clock open.

In [ ]:
!sed -n '1,40p' demos/bitemporal/macros/bitemporal_load.sql
# The whole strategy fits on a screen: close the system clock, carry the superseded business version forward, insert the new rows.

| The question | Which clock | The filter |
|---|---|---|
| What is current? | both open | `dl_current_ind` |
| What was true on date D, as we know it today? | business covers D, system open | `dl_business_start_dt <= D and coalesce(dl_business_end_dt, '9999-12-31') >= D and dl_system_end_ts is null` |
| What did the report say at time T? | system covers T | `dl_system_start_ts <= T and coalesce(dl_system_end_ts, '9999-12-31') > T` |
| Which versions did a correction replace? | same business start, later system start | order by `dl_system_start_ts` within the key |

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 GUEST OR VOLUNTEER BRIEF — ten minutes</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Show one real bitemporal table: the DL columns, one row's history ordered by system start, one change that created a version, one correction if you have one, and which strategy the model uses (full, incremental, balance). Then take two questions. If a defect once created a version, show it: it is the best illustration of 'the table is an audit of what the code did'.</div></div>

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 SHAREBACK — which clock?</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">Each table takes one: (1) 'What was the associate's title on 2 November?' (2) 'What did the dashboard show on the morning of day 2?' (3) 'When did we learn about the correction?' (4) 'Which rows does an auditor need to see?' Say the clock, then the filter.</div></div>

<a id="orchestration"></a>

## Orchestration and Friday

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Yesterday's project builds when you type dbt build. Who types it at six in the morning?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Orchestration: deciding when the pipeline's parts run, in what order, with retries, logs and alerts.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>A product's promises about delivery time and freshness are kept by the scheduler, not by the SQL.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Read a DAG as a drawing; then the demo stack runs Day 3's project on a schedule.</div></div></div>

![The challenge's DAG: start, a freshness gate, the dbt task group, publish, end, with callbacks](../common/diagrams/png/d4-dag.png)

- The freshness gate runs first so a stale feed fails the run before any compute is spent.
- Cosmos renders the dbt project as one task per model and test, in dbt's own order.
- Callbacks live in configuration, not in the DAG's work: on failure, on retry, on SLA miss.
- The schedule is a cron in a time zone; catchup is off; retries have backoff.

In [ ]:
!sed -n '/^with DAG/,/^) as dag:/p' demos/airflow_local/dags/eod_prices/dags/eod_prices_daily.py
!grep -n '>>' demos/airflow_local/dags/eod_prices/dags/eod_prices_daily.py
# Day 2's generated product, orchestrated: the schedule is the contract's 07:30 ET commitment, 06:30 weekdays New York. Read >> as 'and then'.

<div style="border:2px solid #D9EAD3; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🗣️ SAY</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">“Friday: you get CSV files about a fictional wealth manager called Atlas and a README. Phase 1 is the certification bar: sources, the three layers, tests, and understanding the orchestration. Two to three hours for most teams. Phase 2 is fourteen extensions you choose from. We watch you build it; nobody hands out solutions.”<br>“Work in data_products/atlas_elt, not in the challenge folder. If you run it in the wrong folder, that is on you, and now you have been told.”</div></div>

<a id="airflow"></a>

## Airflow

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">What is actually running when you type astro dev start?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>A DAG processor, a scheduler, workers, a triggerer, an API server with the UI, and a metadata database: on your laptop as containers, or hosted by Astronomer.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Knowing the pieces explains the symptoms: an import error is the processor; a queued task is the scheduler; a failed task is a worker log.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>Trigger a run, read the graph and the grid, open a task log, read the code tab, break it on purpose, fix it.</div></div></div>

![Airflow's components, locally and hosted](../common/diagrams/png/d4-airflow-components.png)

- The DAG processor parses dags/ and reports import errors; the scheduler decides what is due; workers run tasks; the triggerer handles deferred waits; the API server serves the UI.
- astro dev start runs all of them as containers on the laptop; a hosted Astro deployment is the same thing run for you and vended per team.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; the stack, a run, the UI</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">
         In a terminal in <code>demos/airflow_local/</code>: <code>bash run.sh start</code> (done before the day), then the cells below: seed, trigger, wait, status. Open <a href="http://localhost:8080">localhost:8080</a>: the DAGs list (owner, schedule, tags), the graph (five boxes and the dbt task group), the grid (one column per run), a task log (click <code>publish_to_catalog</code>: the published rows are in the log), the code tab.<br>
         Same commands as the learners': <code>astro dev run dags list</code>, <code>astro dev run dags trigger …</code>, <code>astro dev run tasks states-for-dag-run …</code>.
         </div></div>

In [ ]:
!bash demos/airflow_local/run.sh seed 2>&1 | grep -E 'Done|rror'

In [ ]:
!bash demos/airflow_local/run.sh trigger && bash demos/airflow_local/run.sh wait

In [ ]:
!bash demos/airflow_local/run.sh status | tail -30

In [ ]:
!sed -n '/^default_args/,/^}/p' demos/airflow_local/dags/portfolio_analytics/dags/portfolio_analytics_daily.py
!grep -n 'def on_' demos/airflow_local/dags/portfolio_analytics/dags/callbacks.py
# default_args are inherited by every task: retries with backoff, the failure and retry callbacks. The SLA-miss callback sits on the DAG.

![Astro blueprints: YAML in, a DAG out](../common/diagrams/png/d4-blueprints.png)

- A blueprint is Python written once by the platform team with placeholders; a dataset's YAML fills them; the result is a registered DAG that triggers a Databricks job.
- Over ninety percent of products fit a blueprint; a custom DAG is for the rest, and can become a new blueprint.
- The challenge is Python on purpose: once the Python makes sense, the YAML is just configuration.

In [ ]:
!sed -n '1,30p' ../../reference/DataCreators-client/dev-container/edo-training-data-products/data_products/.blueprints/dag_exec_check.py
# A platform blueprint from the vended repository: the Python learners never write, filled from YAML.

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; a controlled failure, then the repair</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">Make the custodian feed look two days old, trigger: the freshness gate fails and everything downstream is <em>upstream_failed</em>; no publish, no rows. Open the failed task's log and read the first useful line. Then stamp the feed fresh, trigger, green. Say: this is the same drill as the challenge's Phase 2 and Friday's extension questions: late input, failed check, repeated run.</div></div>

In [ ]:
!bash demos/airflow_local/run.sh stale && bash demos/airflow_local/run.sh trigger && bash demos/airflow_local/run.sh wait && bash demos/airflow_local/run.sh status | grep -E 'source_freshness|publish|state' | head -6

In [ ]:
!bash demos/airflow_local/run.sh fresh && bash demos/airflow_local/run.sh trigger && bash demos/airflow_local/run.sh wait

<a id="challenge"></a>

## Challenge 10

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🏆 CHALLENGE TIME &nbsp;—&nbsp; Airflow practical (challenge 03) &nbsp;·&nbsp; until 13:30, with lunch</div>
<div style="background:#f8f8f8; padding:12px 14px; color:#1a1a1a;">In the training repository: <code>challenges/v1/03-airflow-orchestration/README.md</code>.<br><br>
         <strong>Prerequisites, in this order:</strong> <code>mise run setup:databricks</code> done; <code>(cd challenges/v1/02-dbt-pipeline/dbt && dbt seed)</code> (the DAG reads the same four feeds, refreshed); then, from the repo root, <code>astro dev start</code> <em>after</em> setup finished. The UI address is the one astro prints (a per-project hostname and port), not a fixed localhost:8080.<br>
         <strong>Phase 1</strong> (about 1.5 hours): 1 why orchestration · 2 anatomy of a DAG · 3 <code>dbt debug</code> · 4 the freshness gate (<code>astro dev run tasks test portfolio_analytics_daily source_freshness_check 2026-05-04</code>) · 5 the Cosmos task group · 6 wire the chain (the one TODO: <code>start >> source_freshness_check >> dbt_pipeline >> publish_to_catalog >> end</code>) · 7 schedule · 8 retries · 9 first full run · 10 logs and re-running.<br>
         <strong>Phase 2:</strong> the alert transport, tiered alerting, SLA and retry tiers, wiring callbacks, the freshness gate as a contract, the monitoring config, the runbook and consumer guide, structural tests, the Databricks submit-run variant, a production-ready DAG.<br>
         <strong>Gate:</strong> <code>mise run challenge:verify 03</code> (a static check of the wiring); the live run is the evidence: graph, logs, states.<br>
         <strong>Finished early?</strong> What happens if the input is late? If the quality check fails? If the run is repeated? Show a log or output check that supports each answer.
         </div></div>

| Symptom | Cause | Fix |
|---|---|---|
| `dags list-import-errors` shows a Cosmos error ending in an empty catalog name | Astro started before Databricks setup wrote the env | New terminal where `echo $DBT_CATALOG` prints a value, then `astro dev restart --wait 5m` |
| The freshness gate fails on a fresh seed | Connection error, not staleness: the task now says which | Same cold-start fix; then re-run the task |
| `airflow: command not found` | The CLI lives in the containers | Prefix with `astro dev run` |
| `astro dev parse` says 0 DAGs | It does not mount data_products | Use the README's `python -c "import ast…"` check and `astro dev run dags list-import-errors` |
| A BashOperator DAG is rejected | CG's cluster policy blocks arbitrary-shell operators | PythonOperator calling subprocess, as the starter does |
| Tasks run one at a time | `max_active_tasks` (learners: 4; this laptop: 1 for DuckDB) | Expected; explain why |
| `dbt docs serve` fails to bind | `DBT_HOST` is the Databricks host | `--host 127.0.0.1 --port 8081` |
| Two Astro environments on one laptop | Another project is running | `docker ps` and look for the devcontainer's scheduler |

<a id="cicd"></a>

## CI/CD and deployment

<div style="border:2px solid #434343; border-radius:8px; margin:18px 0 12px; overflow:hidden;">
<div style="background:#434343; color:#fff; padding:8px 14px; font-weight:bold;">🧭 WHAT, WHY AND HOW</div>
<div style="background:#F3F3F3; padding:10px 14px; color:#1a1a1a; font-style:italic;">Your DAG and dbt project are in Git. How do they get to production, and who presses the button?</div>
<div style="display:flex; gap:8px; padding:10px; background:#fff;"><div style="flex:1; min-width:0; background:#D0E0E3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHAT</div>Continuous integration (merge often) and continuous deployment (release often), driven by GitHub events through Harness, into Astro and Databricks, watched by Datadog.</div><div style="flex:1; min-width:0; background:#FFF2CC; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">WHY</div>Creators rarely operate this layer, but they read it: which pipeline ran, where the logs are, who deployed last night.</div><div style="flex:1; min-width:0; background:#D9EAD3; border-radius:6px; padding:10px 12px; color:#1a1a1a;"><div style="font-weight:bold; font-size:0.78em; letter-spacing:0.08em; margin-bottom:4px;">HOW</div>The GitOps picture; a Harness walkthrough (guest, screenshots, or the anchor); monitors as the SLIs made visible.</div></div></div>

![From a branch to production: pull request, merge, Harness, SQA, production with a change record](../common/diagrams/png/d4-cicd.png)

- Integration is merging; continuous means daily or hourly, not monthly. Deployment is releasing to an environment.
- A merge fires a webhook; Harness builds, tests and deploys; the SQA environment is shared, so check who deployed last; production needs a change record.
- GitOps: the Git history becomes the history of what was live. CG is 'halfway': events trigger pipelines, with an approval step between.

<details><summary><strong>🖊️ Whiteboard — CI/CD</strong> (ask, write what the room says, add what is missing)</summary>
<table style="width:100%;border-collapse:collapse;margin:8px 0;color:#1a1a1a"><tr><th style="text-align:left;padding:6px 10px;background:#F3F3F3">Ask</th><th style="text-align:left;padding:6px 10px;background:#F3F3F3">Expect, then add</th></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>What do CI and CD mean?</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Integration: merging a feature branch, often. Deployment: releasing to an environment, including production, often.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>Why bother?</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Faster feedback that something is broken; a shared understanding of the product across the team; observability of the lifecycle.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>What are the downsides?</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>Automation without a real human in the loop; fragility; correlated failures (when one shared system is down, everything is); the maintenance, approval and audit burden. It is not free.</td></tr><tr><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc;width:38%'><strong>What is Harness here?</strong></td><td style='vertical-align:top;padding:6px 10px;border-top:1px solid #ccc'>The CI/CD system, roughly GitOps: a pull request submitted and approved triggers the pipeline. It isolates the creator from the deployment machinery.</td></tr></table></details>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#C9DAF8; color:#1a1a1a; padding:8px 14px; font-weight:bold;">🖥️ DEMO &nbsp;—&nbsp; Harness, with screenshots or a guest</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">
         Walk the real thing if someone has access, else the screenshots collected in the pre-flight: pick the environment; find the repository's pipeline; <strong>SQA</strong>: Run → apply the SQA input set → choose the feature branch → run; a few minutes; check who deployed last and whether they finished testing, because SQA is shared. <strong>Production</strong>: the PRD pipeline, fixed values, the release branch, and a change record (CRQ) number before anything runs. Logs for a failed execution. Then: secrets live in AWS Secrets Manager, never in code or a chat; an API token to trigger deployments from an AI assistant is a risk, not a shortcut.
         </div></div>

In [ ]:
!sed -n '1,40p' demos/airflow_local/dags/portfolio_analytics/include/monitoring/portfolio_analytics.yml
# The challenge's monitoring config: six panels, each an SLI with an alert condition and a channel. This is what a Datadog monitor is for: the promises from Day 2, watched.

<a id="exercises"></a>

## Release decision and the failure note

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 10 &nbsp;—&nbsp; Release decision &nbsp;·&nbsp; Pairs &nbsp;·&nbsp; 20 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>The nightly build produced a candidate output with 500 rows but only 497 distinct IDs, so the uniqueness check failed and the candidate was not published. Yesterday's accepted output is still available to consumers. In ten minutes, decide what should happen next and draft a two-sentence update to the consumer. Then ten minutes of discussion.</strong><br><br><strong>Produce:</strong><br>• A decision: what is released, what stays blocked, and who decides<br>• A two-sentence update to the consumer<br>• The information you are missing and who can supply it<br><br><strong>Debrief questions:</strong><br>• Does the consumer's freshness commitment allow yesterday's output to stand today?<br>• What would make the three duplicate IDs a data problem rather than a pipeline problem?<br>• Who gets to override a failed quality gate, and how is that recorded?</div></div>

<details><summary><strong>Discussion notes for Activity 10</strong></summary>

- The failed candidate stays blocked: three IDs appear twice, and until someone knows why, publishing it would publish a lie about uniqueness.
- Yesterday's accepted output is acceptable only if the consumer's freshness commitment allows a one-day-old sheet; the contract says, not the engineer.
- Missing information: which three IDs, whether the source re-delivered, whether the dedup rule or the source changed, who owns the source. The consumer update names what is known, what is not, and when the next update comes.
- Who overrides a failed gate, and where that decision is recorded, is a governance question worth leaving open in the room.

</details>

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#F8DB59; color:#000; padding:8px 14px; font-weight:bold;">🎯 ACTIVITY 11 &nbsp;—&nbsp; Explain a failure and repair &nbsp;·&nbsp; Pairs, then swap &nbsp;·&nbsp; 15 min</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;"><strong>Write a short note about the Airflow failure and repair you ran today (or the prepared example if your environment was blocked): the affected run, the useful error, the cause, the action taken, the output check, and anything still unresolved. Swap notes with another pair for five minutes: could they locate the run and explain what happened from the note alone?</strong><br><br><strong>Produce:</strong><br>• The note, using the template below<br>• The other pair's verdict: could they find the run?<br><br><strong>Debrief questions:</strong><br>• Which line of the note was the most useful to the reader?<br>• What did you have to go back to the UI to find?</div></div>

<details><summary><strong>Prepared failure example</strong> (for pairs whose environment was blocked)</summary>

Expected: the freshness gate fails because the custodian feed is older than its `error_after` threshold; `dbt_pipeline` and `publish_to_catalog` are `upstream_failed`; no rows are published; the previous run's output stands. Observed in the demo stack: the `source_freshness_check` log ends with "dbt source freshness failed: the custodian feed breached its error_after threshold"; the repair was to refresh the feed and trigger a new run, which published eight rows. Distinguish this from a credentials or path error, whose first useful line names a missing environment value or executable.

</details>

<a id="friday"></a>

## Friday preparation

<div style="border:2px solid #F8DB59; border-radius:8px; margin:10px 0;">
<div style="background:#FFF2CC; color:#000; padding:8px 14px; font-weight:bold;">💭 FRIDAY CHECKLIST — every learner, before leaving</div>
<div style="background:#fffef5; padding:12px 14px; color:#1a1a1a;">
         ☐ Open <code>challenges/v1/04-atlas-capstone/README.md</code> and read the plan of the day.<br>
         ☐ From the repo root: <code>test -f .mise.toml && test -d data_products/atlas_elt && echo OK</code> and <code>mise exec -- databricks auth describe --profile DEFAULT</code>.<br>
         ☐ From <code>data_products/atlas_elt</code>: <code>dbt debug</code> passes.<br>
         ☐ <code>astro dev stop</code> so tomorrow starts clean.<br>
         ☐ Any failure above goes on the list with a name and a time it will be fixed.
         </div></div>

<div style="border:2px solid #434343; border-radius:8px; margin:10px 0;">
<div style="background:#D9EAD3; color:#1a1a1a; padding:8px 14px; font-weight:bold;">✅ BEFORE LEARNERS LEAVE</div>
<div style="background:#fff; padding:12px 14px; color:#1a1a1a;">☐ The DAG, run and task identities, useful logs, an output check and a short explanation of the failure and repair, per pair.<br>☐ Every learner has opened challenges/v1/04-atlas-capstone/README.md and run the readiness checks in data_products/atlas_elt.<br>☐ A list of setup issues that could stop Friday's start, each with the person responsible for the next action.<br>☐ The Airflow stack stopped on the learners' laptops (astro dev stop) so Friday starts clean.</div></div>

<a id="questions"></a>

## Questions from the room

Type the question in the table as it comes; answer live in the cells below; tidy up at the break. Anything unanswered goes to the platform team or the Teams channel, with the person who asked.

| # | Question | Who asked | Answer / where it went |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

In [ ]:
# Answer live here: SQL against a demo warehouse, a small Python check, or a shell command.
# Example: sql("select * from training_demo.stg_holdings limit 5", HERE / "demos/portfolio_analytics/warehouse/training_dev.duckdb")

In [ ]:
# Sketch a diagram live (before Excalidraw). Edit and run.
quick_diagram("""
flowchart LR
  A[source file] --> B[dbt rules on Databricks] --> C[consumer table]
""")

**Answers written up after the day** — fill in here, then share the notebook.

- 

---
<small>&copy; 2026 Decoded Limited. All rights reserved. [decoded.com](https://decoded.com)</small>